In [0]:
import sys
import os
from pyspark.sql.functions import col, from_json

sys.path.append(os.path.abspath('.'))
from config import kafka_spark_options
from schemas import flight_schema

checkpoint_path = "/Volumes/workspace/flights_pipeline/checkpoints/flight_stream_checkpoint"
table_name = "workspace.flights_pipeline.incoming_flights"

raw_stream_df = (
    spark.readStream
    .format("kafka")
    .options(**kafka_spark_options)
    .load()
)

parsed_df = (
    raw_stream_df
    .selectExpr("CAST(value AS STRING) as json_string")
    .select(from_json(col("json_string"), flight_schema).alias("data"))
    .select("data.*")
    .filter(col("longitude").isNotNull() & col("latitude").isNotNull())
)

query = (
    parsed_df
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(table_name)
)

query.awaitTermination()
